# Source-conditioned likelihood pilot V30
ATLAS method: Ko et al., https://arxiv.org/abs/2610.19583. Independent implementation. Public pool: Nishant Dahal and original contributors, CC BY-SA 4.0. Fresh Qwen2.5-1.5B base; no feasibility checkpoint or host validation labels used in training. Fixed final epoch; no submission.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[name]='2'
os.environ['TOKENIZERS_PARALLELISM']='false'
Path('source_likelihood.py').write_text('"""Independent implementation of source-conditioned likelihood modeling.\n\nMethod credit: Ko et al., ATLAS, arXiv:2610.19583, equations in sections 3/B.1.\nAuthors\' code reviewed at 93c1b15da6143161e222eecee7ebcdc3c60645a9;\nno source copied. This module uses plain PyTorch adapters, not PEFT internals.\nRouting must stay unchanged until backward completes (including recomputation).\n"""\nimport math\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass SourceRoute:\n    ids: torch.Tensor | None = None\n\n\nclass SourceProjection(nn.Module):\n    def __init__(self, linear, route, sources, rank=16, alpha=32, dropout=.05):\n        super().__init__()\n        self.base = linear.requires_grad_(False)\n        self.route = route\n        self.factor = alpha/rank\n        self.dropout = nn.Dropout(dropout)\n        self.down = nn.Parameter(torch.empty(rank, linear.in_features, device=linear.weight.device))\n        self.up = nn.Parameter(torch.empty(sources, rank, linear.out_features, device=linear.weight.device))\n        nn.init.kaiming_uniform_(self.down, a=math.sqrt(5))\n        nn.init.normal_(self.up, std=1e-4)\n\n    def forward(self, x):\n        ids = self.route.ids\n        if ids is None or ids.shape != (len(x),):\n            raise ValueError(\'Missing or stale source routing\')\n        output = self.base(x)\n        projected = F.linear(self.dropout(x.to(self.down.dtype)), self.down)\n        update = torch.bmm(projected, self.up[ids])\n        return output + (self.factor*update).to(output.dtype)\n\n\ndef clipped_evidence(log_probs, mask, temperatures=(1., 2., 4., 8.)):\n    """[batch, source, token] -> [batch, source]; padding contributes nothing."""\n    if log_probs.ndim != 3 or mask.shape != log_probs[:,0].shape:\n        raise ValueError(\'Invalid token evidence dimensions\')\n    if (mask.sum(-1) == 0).any():\n        raise ValueError(\'Every passage needs a scored token\')\n    lp = log_probs.float()\n    disadvantage = (lp[:,None,:,:]-lp[:,:,None,:]).clamp_min(0).sum(2)\n    evidence = torch.stack([torch.exp(-disadvantage/t) for t in temperatures]).mean(0)\n    return (evidence*mask[:,None,:]).sum(-1)/mask.sum(-1)[:,None]\n\n\nclass SourceLanguageModel(nn.Module):\n    def __init__(self, causal_lm, sources, rank=16, alpha=32, dropout=.05, head_chunk=32):\n        super().__init__()\n        self.lm = causal_lm.requires_grad_(False)\n        self.sources, self.head_chunk = sources, head_chunk\n        self.route = SourceRoute()\n        self.lm.config.use_cache = False\n        names = [name for name, module in self.lm.named_modules()\n                 if isinstance(module, nn.Linear) and name.rsplit(\'.\',1)[-1] in\n                 {\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\'}]\n        if len(names) != 4*self.lm.config.num_hidden_layers:\n            raise ValueError(\'Unexpected attention projection layout\')\n        for name in names:\n            parent_name, attr = name.rsplit(\'.\',1)\n            parent = self.lm.get_submodule(parent_name)\n            setattr(parent, attr, SourceProjection(getattr(parent, attr),self.route,\n                                                  sources,rank,alpha,dropout))\n        embedding = self.lm.get_input_embeddings().weight.detach().float()\n        initial = embedding.mean(0)[None,:].expand(sources,-1).clone()\n        initial += .01*embedding.std()*torch.randn_like(initial)\n        self.prefix = nn.Parameter(initial)\n\n    def token_log_probs(self, token_ids, mask, sources):\n        if sources.shape != (len(token_ids),) or ((sources<0)|(sources>=self.sources)).any():\n            raise ValueError(\'Invalid source IDs\')\n        if token_ids.shape != mask.shape or token_ids.shape[1] < 2:\n            raise ValueError(\'Invalid token batch\')\n        self.route.ids = sources\n        embedding = self.lm.get_input_embeddings()(token_ids)\n        prefix = self.prefix[sources,None,:].to(embedding.dtype)\n        inputs = torch.cat((prefix,embedding),dim=1)\n        attention = torch.cat((torch.ones_like(mask[:,:1]),mask),dim=1)\n        hidden = self.lm.get_decoder()(inputs_embeds=inputs,attention_mask=attention,\n                                      use_cache=False).last_hidden_state[:,1:-1,:]\n        target = token_ids[:,1:]\n\n        def score_chunk(h, target_chunk):\n            logits = self.lm.get_output_embeddings()(h).float()\n            return -F.cross_entropy(logits.flatten(0,1),target_chunk.flatten(),\n                                    reduction=\'none\').view_as(target_chunk)\n\n        parts = []\n        for start in range(0,hidden.shape[1],self.head_chunk):\n            h, target_chunk = hidden[:,start:start+self.head_chunk],target[:,start:start+self.head_chunk]\n            value = checkpoint(score_chunk,h,target_chunk,use_reentrant=False) if self.training else score_chunk(h,target_chunk)\n            parts.append(value)\n        return torch.cat(parts,dim=1)\n\n    def forward(self, token_ids, mask, sources):\n        lp = self.token_log_probs(token_ids,mask,sources)\n        valid = mask[:,1:].float()\n        if (valid.sum(-1)==0).any(): raise ValueError(\'Empty scored passage\')\n        return -((lp*valid).sum(-1)/valid.sum(-1)).mean()\n\n    @torch.no_grad()\n    def all_source_scores(self, token_ids, mask):\n        if self.training: raise ValueError(\'Call eval before scoring all sources\')\n        lp = torch.stack([self.token_log_probs(token_ids,mask,\n                         torch.full((len(token_ids),),k,device=token_ids.device,dtype=torch.long))\n                         for k in range(self.sources)],dim=1)\n        valid = mask[:,1:].float()\n        mean = (lp*valid[:,None,:]).sum(-1)/valid.sum(-1)[:,None]\n        return {\'mean\':mean,\'clipped\':clipped_evidence(lp,valid),\'token_log_probs\':lp}\n\n    def adapter_state(self):\n        return {name:parameter.detach().cpu().clone() for name,parameter in self.named_parameters()\n                if parameter.requires_grad}\n\n    def load_adapter_state(self, state):\n        expected = {name:p for name,p in self.named_parameters() if p.requires_grad}\n        if set(state) != set(expected): raise ValueError(\'Adapter keys differ\')\n        with torch.no_grad():\n            for name, value in state.items():\n                if value.shape != expected[name].shape or not torch.isfinite(value).all():\n                    raise ValueError(\'Invalid adapter tensor: \'+name)\n                expected[name].copy_(value)\n')
Path('likelihood_batches.py').write_text('"""Exact-epoch data-parallel batches without duplicating final-window samples."""\nimport math\nimport numpy as np\n\n\ndef epoch_windows(size, seed, epoch, rank, world_size, batch_size, accumulation):\n    if size < 1 or not 0 <= rank < world_size:\n        raise ValueError(\'Invalid dataset size or rank\')\n    if min(world_size,batch_size,accumulation)<1:\n        raise ValueError(\'Batch dimensions must be positive\')\n    order=np.random.default_rng(np.random.SeedSequence([seed,epoch])).permutation(size)\n    global_microbatch=world_size*batch_size\n    window_size=global_microbatch*accumulation\n    for start in range(0,size,window_size):\n        window=order[start:start+window_size]\n        microsteps=math.ceil(len(window)/global_microbatch)\n        steps=[]\n        for microstep in range(microsteps):\n            offset=microstep*global_microbatch+rank*batch_size\n            indices=window[offset:offset+batch_size]\n            # DDP averages gradients across ranks. Multiplication by world_size\n            # restores the global document-mean gradient, including short tails.\n            weight=world_size*len(indices)/len(window)\n            steps.append((indices,weight,microstep==microsteps-1))\n        yield steps\n')
Path('train_likelihood_v30.py').write_text('"""Fixed two-stage, two-GPU generative attribution pilot; no model selection."""\nimport hashlib\nimport json\nimport math\nimport os\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport time\n\nos.environ[\'HF_HUB_OFFLINE\']=\'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\']=\'1\'\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\nfrom safetensors.torch import save_file,load_file\nfrom transformers import AutoModelForCausalLM,AutoTokenizer\nfrom source_likelihood import SourceLanguageModel\nfrom likelihood_batches import epoch_windows\n\n\ndef main():\n    started=time.monotonic()\n    plan=json.loads(Path(\'plan.json\').read_text())\n    assert plan[\'feasibility_audit\'][\'verified\'] and plan[\'feasibility_audit\'][\'qualifies_for_pilot\']\n    rank=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(rank)\n    dist.init_process_group(\'nccl\')\n    world=dist.get_world_size();assert world==plan[\'world_size\']==2\n    def log(**values):\n        values.update(rank=rank,elapsed_seconds=time.monotonic()-started)\n        with Path(f\'progress-rank{rank}.jsonl\').open(\'a\') as f:f.write(json.dumps(values)+\'\\n\')\n        print(json.dumps(values),flush=True)\n    torch.manual_seed(plan[\'training_seed\'])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\n    frame=pd.read_parquet(\'training_frame.parquet\')\n    valid=pd.read_parquet(\'validation_frame.parquet\')\n    assert len(frame)==20159 and len(valid)==540\n    assert set(frame.loc[frame.stage==\'host\',\'group\']).isdisjoint(valid.group)\n    labels=np.array(plan[\'labels\']);assert len(labels)==18\n    tok=AutoTokenizer.from_pretrained(plan[\'model_dir\'],local_files_only=True)\n    tok.padding_side=\'right\'\n    if tok.pad_token_id is None:tok.pad_token=tok.eos_token\n    def tokenize(data):\n        tokens=tok(data.text.tolist(),truncation=True,max_length=plan[\'max_length\']-2,add_special_tokens=False)[\'input_ids\']\n        return [[tok.eos_token_id]+ids+[tok.eos_token_id] for ids in tokens]\n    train_tokens=tokenize(frame);valid_tokens=tokenize(valid)\n    def batch(tokens,indices):\n        return tok.pad({\'input_ids\':[tokens[i] for i in indices]},padding=True,return_tensors=\'pt\').to(\'cuda\')\n    backbone=AutoModelForCausalLM.from_pretrained(plan[\'model_dir\'],torch_dtype=torch.float16,\n                attn_implementation=\'sdpa\',local_files_only=True).cuda()\n    model=SourceLanguageModel(backbone,18,plan[\'rank\'],plan[\'alpha\'],plan[\'dropout\'],plan[\'head_chunk\'])\n    model.lm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.train();parallel=DistributedDataParallel(model,device_ids=[rank])\n    parameters=[p for p in model.parameters() if p.requires_grad]\n    if rank==0:save_file(model.adapter_state(),\'initial-adapter.safetensors\')\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=plan[\'initial_loss_scale\'],growth_interval=plan[\'loss_scale_growth_interval\'])\n    targets=torch.tensor(frame.label.to_numpy(),device=\'cuda\')\n    total_skipped=0;total_updates=0\n    log(event=\'loaded\',trainable_parameters=sum(p.numel() for p in parameters),\n        training_rows=len(frame),validation_rows=len(valid),device=torch.cuda.get_device_name())\n    for stage in [\'pool\',\'host\']:\n        stage_rows=np.flatnonzero(frame.stage.to_numpy()==stage)\n        epochs=plan[stage+\'_epochs\'];lr=plan[stage+\'_lr\']\n        updates_per_epoch=math.ceil(len(stage_rows)/(world*plan[\'batch_size\']*plan[\'gradient_accumulation\']))\n        stage_updates=epochs*updates_per_epoch;warmup=max(1,math.ceil(stage_updates*plan[\'warmup_fraction\']))\n        optimizer=torch.optim.AdamW(parameters,lr=lr,weight_decay=plan[\'weight_decay\'])\n        stage_step=0\n        for epoch in range(epochs):\n            loss_sum=0.;examples=0\n            windows=epoch_windows(len(stage_rows),plan[\'training_seed\'],epoch+(100 if stage==\'host\' else 0),\n                                  rank,world,plan[\'batch_size\'],plan[\'gradient_accumulation\'])\n            for window_index,steps in enumerate(windows):\n                if time.monotonic()-started>plan[\'wall_limit_seconds\']:raise RuntimeError(\'Training wall limit exceeded\')\n                stage_step+=1\n                factor=stage_step/warmup if stage_step<=warmup else .5*(1+math.cos(math.pi*(stage_step-warmup)/max(stage_updates-warmup,1)))\n                for group in optimizer.param_groups:group[\'lr\']=lr*factor\n                optimizer.zero_grad(set_to_none=True)\n                for local,weight,synchronize in steps:\n                    selected=stage_rows[local] if len(local) else stage_rows[:1]\n                    data=batch(train_tokens,selected)\n                    with (nullcontext() if synchronize else parallel.no_sync()):\n                        with torch.autocast(\'cuda\',dtype=torch.float16):\n                            raw=parallel(data[\'input_ids\'],data[\'attention_mask\'],targets[selected])\n                            loss=raw*weight\n                        assert torch.isfinite(raw)\n                        scaler.scale(loss).backward()\n                    loss_sum+=float(raw.detach())*len(local);examples+=len(local)\n                scaler.unscale_(optimizer)\n                norm=torch.nn.utils.clip_grad_norm_(parameters,plan[\'gradient_clip\'])\n                old=scaler.get_scale();scaler.step(optimizer);scaler.update()\n                skipped=scaler.get_scale()<old;total_skipped+=int(skipped);total_updates+=1\n                if skipped or window_index%50==0:\n                    log(event=\'update\',stage=stage,epoch=epoch,window=window_index,gradient_norm=float(norm),\n                        skipped=skipped,scale=scaler.get_scale(),lr=lr*factor)\n                assert total_skipped<=max(2,int(.01*total_updates)), \'Excess numerical skips\'\n            sums=torch.tensor([loss_sum,examples],device=\'cuda\',dtype=torch.float64)\n            dist.all_reduce(sums)\n            assert int(sums[1])==len(stage_rows)\n            log(event=\'epoch_complete\',stage=stage,epoch=epoch,mean_loss=float(sums[0]/sums[1]),rows=int(sums[1]))\n        name=\'external-only\' if stage==\'pool\' else \'final\'\n        if rank==0:\n            state=model.adapter_state();assert all(torch.isfinite(v).all() for v in state.values())\n            save_file(state,name+\'-adapter.safetensors\')\n        dist.barrier()\n    # Evaluation starts only after the fixed final epoch; no best-epoch selection.\n    model.eval()\n    selected=np.arange(rank,len(valid),world)\n    clipped=[];means=[]\n    for start in range(0,len(selected),plan[\'batch_size\']):\n        if time.monotonic()-started>plan[\'wall_limit_seconds\']:raise RuntimeError(\'Evaluation wall limit exceeded\')\n        indices=selected[start:start+plan[\'batch_size\']]\n        data=batch(valid_tokens,indices)\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            result=model.all_source_scores(data[\'input_ids\'],data[\'attention_mask\'])\n        clipped.append(result[\'clipped\'].cpu().numpy());means.append(result[\'mean\'].cpu().numpy())\n        if start==0:\n            model.load_adapter_state(load_file(\'final-adapter.safetensors\'))\n            with torch.autocast(\'cuda\',dtype=torch.float16):reloaded=model.all_source_scores(data[\'input_ids\'],data[\'attention_mask\'])\n            torch.testing.assert_close(reloaded[\'clipped\'],result[\'clipped\'],atol=1e-5,rtol=1e-4)\n        if start%40==0:log(event=\'validation\',completed=min(start+plan[\'batch_size\'],len(selected)),total=len(selected))\n    scores=np.concatenate(clipped);mean=np.concatenate(means)\n    assert np.isfinite(scores).all() and np.isfinite(mean).all()\n    np.savez_compressed(f\'validation-rank{rank}.npz\',ids=valid.iloc[selected].id.astype(str).to_numpy(),\n                        labels=labels,clipped=scores,mean=mean)\n    log(event=\'complete\',skipped_steps=total_skipped,optimizer_steps=total_updates,\n        peak_allocated_bytes=torch.cuda.max_memory_allocated())\n    dist.barrier();dist.destroy_process_group()\n\n\nif __name__==\'__main__\':main()\n')
Path('prepare_v18.py').write_text('"""Deterministic training-frame preparation and conservative external overlap screen."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport unicodedata\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\n\ndef prepare(host_path, pool_path, fold_by_prompt, plan, output):\n    output = Path(output)\n    output.mkdir(exist_ok=True, parents=True)\n    host = pd.read_csv(host_path, dtype={"id":str, "prompt_id":str})\n    pool = pd.read_parquet(pool_path)\n    assert hashlib.sha256(Path(host_path).read_bytes()).hexdigest() == plan["host_sha256"]\n    assert hashlib.sha256(Path(pool_path).read_bytes()).hexdigest() == plan["pool_sha256"]\n    assert set(host.prompt_id) == set(fold_by_prompt)\n    labels = sorted(host.model.unique())\n    assert len(labels) == 18 and set(pool.model) <= set(labels)\n    def normalize(s):\n        return re.sub(r"\\s+", " ", unicodedata.normalize("NFKC", str(s))).strip().lower()\n    hashes = pool.text.map(normalize)\n    ambiguous = pool.assign(key=hashes).groupby("key").model.nunique()\n    reject = hashes.isin(set(host.text.map(normalize))) | hashes.isin(ambiguous[ambiguous>1].index) | hashes.duplicated()\n    print(json.dumps({"event":"screen_start", "pool_rows":len(pool)}), flush=True)\n    vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True, dtype=np.float32)\n    X = vectorizer.fit_transform(pd.concat([host.text, pool.text], ignore_index=True))\n    H, P = X[:len(host)], X[len(host):]\n    similarity = np.zeros(len(pool), dtype=np.float32)\n    for start in range(0, len(pool), 512):\n        similarity[start:start+512] = (P[start:start+512] @ H.T).max(axis=1).toarray().ravel()\n    reject |= similarity >= plan["external_host_cosine_exclusion"]\n    clean = pool.loc[~reject]\n    selected = pd.concat([g.sample(min(len(g), plan["external_cap_per_class"]), random_state=plan["seed"])\n                          for _,g in clean.groupby("model")], ignore_index=True)\n    selected["group"] = "external:" + selected.source.astype(str) + ":" + selected.prompt_key.astype(str)\n    selected["stage"] = "pool"\n    host["group"] = "host:" + host.prompt_id\n    host["stage"] = "host"\n    valid_mask = host.prompt_id.map(fold_by_prompt) == plan["pilot_fold"]\n    train = pd.concat([selected, host.loc[~valid_mask]], ignore_index=True)\n    valid = host.loc[valid_mask].reset_index(drop=True)\n    for frame in [train, valid]:\n        frame["label"] = frame.model.map({s:i for i,s in enumerate(labels)}).astype(int)\n    train.to_parquet(output / "training_frame.parquet", index=False)\n    valid.to_parquet(output / "validation_frame.parquet", index=False)\n    audit = {"pool_rows":len(pool), "excluded_rows":int(reject.sum()),\n             "near_host_rows":int((similarity >= plan["external_host_cosine_exclusion"]).sum()),\n             "selected_pool_rows":len(selected), "host_fit_rows":int((~valid_mask).sum()),\n             "host_validation_rows":int(valid_mask.sum()), "labels":labels,\n             "training_frame_sha256":hashlib.sha256((output / "training_frame.parquet").read_bytes()).hexdigest(),\n             "validation_frame_sha256":hashlib.sha256((output / "validation_frame.parquet").read_bytes()).hexdigest()}\n    (output / "data-audit.json").write_text(json.dumps(audit, indent=2))\n    print(json.dumps(audit), flush=True)\n    return labels\n')
plan=json.loads("{\"experiment\": \"improvement-v30\", \"kind\": \"ATLAS-based source-conditioned likelihood pilot\", \"model\": \"qwen-lm/qwen2.5/Transformers/1.5b/1\", \"seed\": 1801, \"training_seed\": 3001, \"pilot_fold\": 0, \"external_cap_per_class\": 1000, \"external_host_cosine_exclusion\": 0.3, \"sources\": 18, \"rank\": 16, \"alpha\": 32, \"dropout\": 0.05, \"max_length\": 512, \"batch_size\": 4, \"gradient_accumulation\": 4, \"head_chunk\": 16, \"pool_epochs\": 3, \"host_epochs\": 2, \"pool_lr\": 0.0001, \"host_lr\": 3e-05, \"weight_decay\": 0.01, \"warmup_fraction\": 0.05, \"gradient_clip\": 1.0, \"wall_limit_seconds\": 24300, \"score\": \"mean of clipped relative token evidence at temperatures 1,2,4,8\", \"fixed_blend_weight\": 0.15, \"blend_transform\": \"log(clip(clipped_score,1e-8,1))\", \"promotion\": {\"fixed_sparse_blend_minimum\": 0.5311111111, \"reference\": \"V22 SupCon fixed sparse blend\", \"require_prompt_bootstrap_gain_lower_positive\": true, \"bootstrap_seed\": 3001, \"bootstrap_samples\": 10000}, \"selection\": \"Evaluate final host epoch only; raw mean likelihood diagnostic cannot replace clipped readout after results\", \"launch_requires\": \"Independent V31 two-GPU audit passes fixed runtime and numerical gates; initialize anew, never use feasibility checkpoint\", \"sources_references\": [\"https://arxiv.org/abs/2610.19583\", \"https://github.com/komseok0109/ATLAS\", \"https://huggingface.co/Qwen/Qwen2.5-1.5B\"], \"attribution\": \"ATLAS method by Ko et al.; independent PyTorch implementation. Public external pool by Nishant Dahal and original contributors, CC BY-SA 4.0. Qwen2.5 backbone by Qwen, Apache 2.0.\", \"limitations\": [\"Repeatedly inspected pilot fold; no untouched-test claim\", \"18 exact source models differ from paper five-family setup\", \"1.5B and smaller batch differ from main paper; not a controlled classifier versus generative ablation\", \"Sparse proxy is not full public ensemble; passing gate authorizes broader validation, not automatic submission\", \"No paid compute\"], \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"world_size\": 2, \"initial_loss_scale\": 128, \"loss_scale_growth_interval\": 1000000, \"maximum_skipped_steps\": \"max(2, floor(0.01 * attempted optimizer updates)); abort above this bound\", \"feasibility_audit\": {\"verified\": true, \"source_sha256\": {\"source_likelihood.py\": \"c816f9f2d9683039bea221cb22520a8daf61959d88a5fec00eba5abbeee7e801\", \"probe_likelihood_v31.py\": \"e42b5da6186a48896b16007bed2ec05fb6f7187fe8fa9a917499a063b976d75a\"}, \"replicas_identical\": true, \"qualifies_for_pilot\": true, \"rank_results\": [{\"verified_updates\": true, \"checkpoint_reload_verified\": true, \"changed_tensors\": 225, \"mean_training_microstep_seconds\": 2.0730496188750074, \"all_source_batch_seconds\": 13.044461765999984, \"projected_pilot_seconds\": 17473.534060008802, \"projected_hidden_seconds\": 18275.01948674998, \"peak_allocated_bytes\": 4269575168, \"qualifies_for_pilot\": true, \"checkpoint_discard_required\": true, \"submission_authorized\": false}, {\"verified_updates\": true, \"checkpoint_reload_verified\": true, \"changed_tensors\": 225, \"mean_training_microstep_seconds\": 2.073005933625012, \"all_source_batch_seconds\": 13.980202631999987, \"projected_pilot_seconds\": 17599.540611446337, \"projected_hidden_seconds\": 19327.727960999986, \"peak_allocated_bytes\": 4269575168, \"qualifies_for_pilot\": true, \"checkpoint_discard_required\": true, \"submission_authorized\": false}], \"checkpoint_discard_required\": true}, \"feasibility_audit_sha256\": \"0a495fbf0962d6ea868a376a7d950376266109c59a1ab3ea54e53681b67ae3b2\", \"source_sha256\": {\"source_likelihood.py\": \"c816f9f2d9683039bea221cb22520a8daf61959d88a5fec00eba5abbeee7e801\", \"likelihood_batches.py\": \"d49b48522019d7a92d37fd5123ec9438740002fc8bc52411504ccad9b5c927a8\", \"train_likelihood_v30.py\": \"70d0dea71a96efde6d960443e6c284ff16b2dae447b487796edb14b4cb8a32c1\", \"prepare_v18.py\": \"84b49ea98456a0a8f95e1925b780003af25dff52d325f668a3e966ddd2e1d0b9\"}}")
fold_by_prompt=json.loads("{\"p000\": 1, \"p001\": 4, \"p002\": 3, \"p003\": 3, \"p004\": 1, \"p005\": 2, \"p006\": 3, \"p007\": 2, \"p008\": 3, \"p009\": 0, \"p010\": 1, \"p011\": 1, \"p012\": 0, \"p013\": 3, \"p014\": 4, \"p015\": 1, \"p016\": 1, \"p017\": 3, \"p018\": 0, \"p019\": 0, \"p020\": 4, \"p021\": 4, \"p022\": 1, \"p023\": 2, \"p024\": 2, \"p025\": 2, \"p026\": 0, \"p027\": 1, \"p028\": 1, \"p029\": 0, \"p030\": 0, \"p031\": 0, \"p032\": 1, \"p033\": 2, \"p034\": 2, \"p035\": 2, \"p036\": 0, \"p037\": 4, \"p038\": 3, \"p039\": 2, \"p040\": 1, \"p041\": 4, \"p042\": 1, \"p043\": 2, \"p044\": 1, \"p045\": 0, \"p046\": 3, \"p047\": 2, \"p048\": 4, \"p049\": 2, \"p050\": 3, \"p051\": 1, \"p052\": 4, \"p053\": 2, \"p054\": 3, \"p055\": 0, \"p056\": 0, \"p057\": 4, \"p058\": 4, \"p059\": 3, \"p060\": 1, \"p061\": 3, \"p062\": 2, \"p063\": 3, \"p064\": 0, \"p065\": 1, \"p066\": 1, \"p067\": 1, \"p068\": 0, \"p069\": 0, \"p070\": 2, \"p071\": 4, \"p072\": 3, \"p073\": 0, \"p074\": 4, \"p075\": 1, \"p076\": 0, \"p077\": 2, \"p078\": 0, \"p079\": 3, \"p080\": 2, \"p081\": 1, \"p082\": 0, \"p083\": 3, \"p084\": 2, \"p085\": 1, \"p086\": 1, \"p087\": 4, \"p088\": 4, \"p089\": 3, \"p090\": 4, \"p091\": 4, \"p092\": 4, \"p093\": 2, \"p094\": 2, \"p095\": 2, \"p096\": 1, \"p097\": 2, \"p098\": 3, \"p099\": 4, \"p100\": 3, \"p101\": 2, \"p102\": 4, \"p103\": 4, \"p104\": 0, \"p105\": 1, \"p106\": 4, \"p107\": 4, \"p108\": 0, \"p109\": 1, \"p110\": 0, \"p111\": 2, \"p112\": 3, \"p113\": 2, \"p114\": 2, \"p115\": 3, \"p116\": 4, \"p117\": 2, \"p118\": 0, \"p119\": 3, \"p120\": 3, \"p121\": 4, \"p122\": 1, \"p123\": 2, \"p124\": 4, \"p125\": 3, \"p126\": 3, \"p127\": 0, \"p128\": 0, \"p129\": 4, \"p130\": 4, \"p131\": 0, \"p132\": 0, \"p133\": 1, \"p134\": 3, \"p135\": 3, \"p136\": 3, \"p137\": 1, \"p138\": 2, \"p139\": 3, \"p140\": 4, \"p141\": 0, \"p142\": 1, \"p143\": 0, \"p144\": 4, \"p145\": 0, \"p146\": 1, \"p147\": 3, \"p148\": 2, \"p149\": 4}")

from prepare_v18 import prepare
hosts=list(Path('/kaggle/input').rglob('train.csv'))
pools=list(Path('/kaggle/input').rglob('pool_v4.parquet'))
assert len(hosts)==len(pools)==1
models=[]
for path in Path('/kaggle/input').rglob('config.json'):
    cfg=json.loads(path.read_text())
    if cfg.get('model_type')=='qwen2' and cfg.get('hidden_size')==1536 and cfg.get('num_hidden_layers')==28:
        models.append(path.parent)
assert len(models)==1
plan['model_dir']=str(models[0])
plan['labels']=prepare(hosts[0],pools[0],fold_by_prompt,plan,'.')
Path('plan.json').write_text(json.dumps(plan,indent=2))
subprocess.run([sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',
                'train_likelihood_v30.py'],check=True)
print(json.dumps({'training_complete':True,'decision':'Independent local fixed-gate evaluation required',
                  'submission_authorized':False}),flush=True)
